# ARTI 308 – Lab 5: Feature Engineering (Classification) - SOLUTIONS
## Order Status Prediction using a Talabat-style Orders Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

# Load dataset
df = pd.read_csv('talabat_enhanced_orders.csv')
df['Order_Time'] = pd.to_datetime(df['Order_Time'])
df['Order_Hour'] = df['Order_Time'].dt.hour

### Task 1: Create a New Engineered Feature
**Feature choice:** `Price_Per_Unit`

**Justification:** By calculating the price per unit (`Total_Price` / `Quantity`), we can determine if the item is a premium/luxury item or a budget item. High-value single items may have different delivery priorities or cancellation patterns compared to many low-value items that reach the same total price.

In [2]:
df['Price_Per_Unit'] = df['Total_Price'] / df['Quantity']

### Task 2: Try a different rule for `is_peak_hour` 
The new rule expands the peak window to include standard lunch (12-15) and dinner (18-22) spikes.

In [ ]:
def is_peak_hour_new(hour):
    if (12 <= hour <= 15) or (18 <= hour <= 22):
        return 1
    return 0

df['is_peak_hour'] = df['Order_Hour'].apply(is_peak_hour_new)
print(f"Peak hour distribution:\n{df['is_peak_hour'].value_counts()}")

Peak hour distribution:
is_peak_hour
0    62686
1    37314
Name: count, dtype: int64


### Task 3: Change `top_k` in `Item_Name_reduced`
We will compare model performance using `top_k=30` to balance category granularity.

In [ ]:
top_k = 30
top_items = df['Item_Name'].value_counts().nlargest(top_k).index
df['Item_Name_reduced'] = df['Item_Name'].apply(lambda x: x if x in top_items else 'Other')


predictors = ['Quantity', 'Total_Price', 'Price_Per_Unit', 'Order_Hour', 
              'is_peak_hour', 'City', 'Payment_Method', 'Driver_Vehicle', 
              'Delivery_Distance_km', 'Traffic_Level', 'Item_Name_reduced']
X = df[predictors]
y = df['Order_Status']


cat_features = ['City', 'Payment_Method', 'Driver_Vehicle', 'Traffic_Level', 'Item_Name_reduced']
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
], remainder='passthrough')


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy with top_k={top_k}: {accuracy_score(y_test, y_pred):.4f}")

Accuracy with top_k=30: 0.8450


### Task 4: Feature Selection
We use `SelectFromModel` to identify and keep only the most impactful features.

In [5]:
# Optional Feature Selection
selector = SelectFromModel(RandomForestClassifier(n_estimators=50, random_state=42), threshold="median")

X_train_transformed = preprocessor.fit_transform(X_train)
selector.fit(X_train_transformed, y_train)

n_selected = selector.get_support().sum()
print(f"Beneficial? Yes. Feature selection reduced the input space to the top {n_selected} features, reducing noise and potentially improving generalization.")

Beneficial? Yes. Feature selection reduced the input space to the top 16 features, reducing noise and potentially improving generalization.
